In [2]:
import pandas as pd
import pyreadstat
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. Define file paths
sav_path_2017 = r"E:\Dissertation 2026\Siyan Xin\2017~2018\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav"
excel_125_path = "125.xlsx"
excel_179_path = "179.xlsx"

# 2. Read and extract activity suffixes from Excel
df_125_suf = pd.read_excel(excel_125_path)
df_179_suf = pd.read_excel(excel_179_path)
suffixes_125 = df_125_suf.iloc[:, 0].dropna().astype(str).tolist()
suffixes_179 = df_179_suf.iloc[:, 0].dropna().astype(str).tolist()

# 3. Read metadata to get column names and dictionaries
print("Reading 2017 metadata...")
_, meta = pyreadstat.read_sav(sav_path_2017, metadataonly=True)
available_columns = set(meta.column_names)

# Dynamically extract numeric codes for the 32 Greater London boroughs
value_labels = meta.variable_value_labels.get('LA_2023', {})
london_codes = []
for code, label in value_labels.items():
    label_str = str(label).strip()
    if label_str.startswith('E09') and 'city of london' not in label_str.lower():
        london_codes.append(float(code))

# 4. Organize base columns to extract (using 2017 specific names: VOLANY, VOLFRQ_POP)
wt_cols = [col for col in available_columns if str(col).startswith('wt_')]
base_cols_raw = [
    'LA_2023', 'Reg9', 'LondInOut', 'Age16plus', 'Age9', 'Disab3', 
    'VOLANY', 'mode', 'serial', 'Number_Activities_150', 'month',
    'MEMS7_ALL', 'MEMS7GR_ALL', 'VOLFRQ_POP'
]
base_cols_raw += [f'disty{i}_POP' for i in range(1, 14)] 
base_cols_raw += [f'volint{i}' for i in range(1, 8)]     
base_cols_raw += wt_cols                                 

# Generate full activity columns
activity_prefixes = ['MEMS7_', 'MEMS7GR_', 'INOUTA_', 'INOUTB_', 'DAYS10P60GR_', 'MONTHS_12_']
activity_cols_125 = [f"{prefix}{suffix}" for prefix in activity_prefixes for suffix in suffixes_125]
activity_cols_179 = [f"{prefix}{suffix}" for prefix in activity_prefixes for suffix in suffixes_179]

# Get the union of columns and ensure they exist in the dataset
all_required_cols = list(set(base_cols_raw + activity_cols_125 + activity_cols_179))
columns_to_read = [col for col in all_required_cols if col in available_columns]

# 5. Extract data rapidly without applying value formats
print(f"Extracting {len(columns_to_read)} columns from 2017 data...")
df_2017, _ = pyreadstat.read_sav(
    sav_path_2017, 
    usecols=columns_to_read, 
    apply_value_formats=False 
)
df_2017['year'] = 2017

# 6. Standardize column names to match team expectations
rename_dict_2017 = {
    'VOLANY': 'VolAny',
    'VOLFRQ_POP': 'VolFrqB_Pop'
}
df_2017.rename(columns=rename_dict_2017, inplace=True)
base_cols_final = [rename_dict_2017.get(col, col) for col in base_cols_raw]

# 7. Replace abnormal/missing codes with NaN (excluding serial and weights)
missing_codes = [-8, -9, 97, 98, 99]
cols_to_recode = [col for col in df_2017.columns if col != 'serial' and not str(col).startswith('wt_')]
df_2017[cols_to_recode] = df_2017[cols_to_recode].replace(missing_codes, np.nan)

# 8. Filter for the 32 London boroughs
df_2017 = df_2017[df_2017['LA_2023'].isin(london_codes)].copy()

# 9. Extract target variables and export to CSV
final_cols_125 = [col for col in (base_cols_final + ['year'] + activity_cols_125) if col in df_2017.columns]
final_cols_179 = [col for col in (base_cols_final + ['year'] + activity_cols_179) if col in df_2017.columns]

print(f"Rows remaining after London filter: {len(df_2017)}")
print(f"Exporting 2017 125-activities CSV ({len(final_cols_125)} columns)...")
df_2017[final_cols_125].to_csv('2017_data_125_activities.csv', index=False)

print(f"Exporting 2017 179-activities CSV ({len(final_cols_179)} columns)...")
df_2017[final_cols_179].to_csv('2017_data_179_activities.csv', index=False)

print("2017 Data extraction completed!")

Reading 2017 metadata...
Extracting 1052 columns from 2017 data...
Rows remaining after London filter: 15967
Exporting 2017 125-activities CSV (753 columns)...
Exporting 2017 179-activities CSV (1053 columns)...
2017 Data extraction completed!


In [5]:
import pandas as pd

# 1. Define file paths (adjust according to your actual saved paths)
csv_125_path = '2017_data_125_activities.csv'
csv_179_path = '2017_data_179_activities.csv'

# 2. Define the 6 feature prefixes for activities
activity_prefixes = ('MEMS7_', 'MEMS7GR_', 'INOUTA_', 'INOUTB_', 'DAYS10P60GR_', 'MONTHS_12_')

def check_column_counts(file_path):
    # Use nrows=0 to load only the column names, saving significant memory and read time
    df = pd.read_csv(file_path, nrows=0)
    all_cols = df.columns.tolist()
    
    # 3. Filter activity columns (columns starting with the 6 prefixes)
    activity_cols = [col for col in all_cols if col.startswith(activity_prefixes)]
    
    # 4. Filter base variable columns (non-activity columns)
    base_cols = [col for col in all_cols if not col.startswith(activity_prefixes)]
    
    # 5. Print statistical results
    print(f"📊 Analyzing file: {file_path}")
    print(f"  ▶ Total columns: {len(all_cols)}")
    print(f"  ▶ Base variable columns count: {len(base_cols)}")
    print(f"  ▶ Activity variable columns count: {len(activity_cols)}")
    print("-" * 40)

# Execute the check
check_column_counts(csv_125_path)
check_column_counts(csv_179_path)

📊 Analyzing file: 2017_data_125_activities.csv
  ▶ Total columns: 753
  ▶ Base variable columns count: 41
  ▶ Activity variable columns count: 712
----------------------------------------
📊 Analyzing file: 2017_data_179_activities.csv
  ▶ Total columns: 1053
  ▶ Base variable columns count: 41
  ▶ Activity variable columns count: 1012
----------------------------------------
